In [ ]:
import pandas as pd
from dataHub import dataHub
from sqlalchemy import text

# Always comment these lines so you have to verify connections prior to executing
# from_con = dataHub('postgre'); to_con = dataHub('cockroach')
# from_con = dataHub('cockroach'); to_con = dataHub('postgre')

# DROP VIEWS

In [31]:
df_views = pd.read_sql("SELECT * FROM information_schema.views WHERE table_name LIKE '%%_vw' ORDER BY table_name", to_con.db_con)
df_views.to_csv('vw_backup.csv', index=False)

db_ex = to_con.db_con.connect()
for _, row in df_views.iterrows():
    db_ex.execute(text(f'DROP VIEW IF EXISTS {row["table_schema"]+"."+row["table_name"]} CASCADE'))
db_ex.commit()

# NBA

INJURIES

In [3]:
# Query from from_com
injuries = pd.read_sql(f"SELECT * FROM nba.injuries WHERE game_date >= '{str(from_con.cur_season_year) + '-06-01'}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM nba.injuries WHERE game_date >= '{str(to_con.cur_season_year) + '-06-01'}'"))
to_ex.commit()

# Write to to_con
injuries.drop_duplicates().to_sql('injuries', to_con.db_con, schema='nba', index=False, if_exists='append')

743

KEY DATES

In [4]:
# Query from from_com
key_dates = pd.read_sql(f"SELECT * FROM nba.key_dates WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM nba.key_dates WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
key_dates.drop_duplicates().to_sql('key_dates', to_con.db_con, schema='nba', index=False, if_exists='append')

4

LEAGUE GAME SCHEDULE

In [5]:
# Query from from_com
league_game_schedule = pd.read_sql(f"SELECT * FROM nba.league_game_schedule WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM nba.league_game_schedule WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
league_game_schedule.drop_duplicates().to_sql('league_game_schedule', to_con.db_con, schema='nba', index=False, if_exists='append')

106

PLAYER BOX SCORE

In [6]:
# Query from from_com
cur_season_game_ids = ', '.join(league_game_schedule['game_id'].astype('str'))
player_box_score = pd.read_sql(f"SELECT * FROM nba.player_box_score WHERE game_id IN ({cur_season_game_ids})", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM nba.player_box_score WHERE game_id IN ({cur_season_game_ids})"))
to_ex.commit()

# Write to to_con
player_box_score.drop_duplicates().to_sql('player_box_score', to_con.db_con, schema='nba', index=False, if_exists='append')

156

PLAYER INFO

In [7]:
# Query from from_com
player_info = pd.read_sql(f"SELECT * FROM nba.player_info WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM nba.player_info WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
player_info.drop_duplicates().to_sql('player_info', to_con.db_con, schema='nba', index=False, if_exists='append')

565

PLAYER SEASON STATS - CONSIDER DELETING THIS TABLE

In [8]:
player_season_stats = pd.read_sql('SELECT * FROM nba.player_season_stats', from_con.db_con)
player_season_stats.drop_duplicates().to_sql('player_season_stats', to_con.db_con, schema='nba', index=False, if_exists='replace')

560

TEAM BOX SCORE

In [9]:
# Query from from_com
team_box_score = pd.read_sql(f"SELECT * FROM nba.team_box_score WHERE game_id IN ({cur_season_game_ids})", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM nba.team_box_score WHERE game_id IN ({cur_season_game_ids})"))
to_ex.commit()

# Write to to_con
team_box_score.drop_duplicates().to_sql('team_box_score', to_con.db_con, schema='nba', index=False, if_exists='append')

128

TEAM ROSTER

In [10]:
# Query from from_com
team_roster = pd.read_sql(f"SELECT * FROM nba.team_roster WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM nba.team_roster WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
team_roster.drop_duplicates().to_sql('team_roster', to_con.db_con, schema='nba', index=False, if_exists='append')

639

TEAMS

In [11]:
teams = pd.read_sql('SELECT * FROM nba.teams', from_con.db_con)
teams.drop_duplicates().to_sql('teams', to_con.db_con, schema='nba', index=False, if_exists='replace')

30

# FTY

CATEGORY LABEL

In [12]:
category_label = pd.read_sql("SELECT * FROM fty.category_label", from_con.db_con)
category_label.drop_duplicates().to_sql('category_label', to_con.db_con, schema='fty', index=False, if_exists='replace')

26

COMPETITOR ROSTER

In [13]:
# Query from from_com
competitor_roster = pd.read_sql(f"SELECT * FROM fty.competitor_roster WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM fty.competitor_roster WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
competitor_roster.drop_duplicates().to_sql('competitor_roster', to_con.db_con, schema='fty', index=False, if_exists='append')

493

FREE AGENTS

In [14]:
free_agents = pd.read_sql('SELECT * FROM fty.free_agents', from_con.db_con)
free_agents.to_sql('free_agents', to_con.db_con, schema='fty', index=False, if_exists='replace')

660

LEAGUE

In [15]:
# Query from from_com
league = pd.read_sql(f"SELECT * FROM fty.league WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM fty.league WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
league.drop_duplicates().to_sql('league', to_con.db_con, schema='fty', index=False, if_exists='append')

4

LEAGUE CATEGORIES

In [16]:
# Query from from_com
league_categories = pd.read_sql(f"SELECT * FROM fty.league_categories WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM fty.league_categories WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
league_categories.drop_duplicates().to_sql('league_categories', to_con.db_con, schema='fty', index=False, if_exists='append')

37

LEAGUE COMPETITOR

In [17]:
# Query from from_com
league_competitor = pd.read_sql(f"SELECT * FROM fty.league_competitor WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM fty.league_competitor WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
league_competitor.drop_duplicates().to_sql('league_competitor', to_con.db_con, schema='fty', index=False, if_exists='append')

46

LEAGUE MATCHUP

In [18]:
# Query from from_com
league_matchup = pd.read_sql(f"SELECT * FROM fty.league_matchup WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM fty.league_matchup WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
league_matchup.drop_duplicates().to_sql('league_matchup', to_con.db_con, schema='fty', index=False, if_exists='append')

806

LEAGUE MATCHUP DATES

In [19]:
# Query from from_com
league_matchup_dates = pd.read_sql(f"SELECT * FROM fty.league_matchup_dates WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM fty.league_matchup_dates WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
league_matchup_dates.drop_duplicates().to_sql('league_matchup_dates', to_con.db_con, schema='fty', index=False, if_exists='append')

70

MATCHUP BOX SCORE

In [20]:
# Query from from_com
matchup_box_score = pd.read_sql(f"SELECT * FROM fty.matchup_box_score WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM fty.matchup_box_score WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
matchup_box_score.drop_duplicates().to_sql('matchup_box_score', to_con.db_con, schema='fty', index=False, if_exists='append')

690

RECENT ACTIVITY

In [21]:
# Query from from_com
recent_activity = pd.read_sql(f"SELECT * FROM fty.recent_activity WHERE season = '{from_con.cur_season}'", from_con.db_con)

# Delete from to_con
to_ex = to_con.db_con.connect()
to_ex.execute(text(f"DELETE FROM fty.recent_activity WHERE season = '{to_con.cur_season}'"))
to_ex.commit()

# Write to to_con
recent_activity.drop_duplicates().to_sql('recent_activity', to_con.db_con, schema='fty', index=False, if_exists='append')

55

# UTIL

In [22]:
nba_fty_name_match = pd.read_sql('SELECT * FROM util.nba_fty_name_match', from_con.db_con)
nba_fty_name_match.to_sql('nba_fty_name_match', to_con.db_con, schema='util', index=False, if_exists='replace')

28

In [14]:
table_column_order = pd.read_sql('SELECT * FROM util.table_column_order', from_con.db_con)
table_column_order.to_sql('table_column_order', to_con.db_con, schema='util', index=False, if_exists='replace')

233

In [24]:
update_log = pd.read_sql('SELECT * FROM util.update_log', from_con.db_con)
update_log.to_sql('update_log', to_con.db_con, schema='util', index=False, if_exists='replace')

277

In [25]:
update_schedule = pd.read_sql('SELECT * FROM util.update_schedule', from_con.db_con)
update_schedule.to_sql('update_schedule', to_con.db_con, schema='util', index=False, if_exists='replace')

9

# ANL

In [26]:
pts_prediction = pd.read_sql('SELECT * FROM anl.pts_prediction', from_con.db_con)
pts_prediction.drop_duplicates().to_sql('pts_prediction', to_con.db_con, schema='anl', index=False, if_exists='replace')

929

# RECREATE VIEWS

In [32]:
db_ex = to_con.db_con.connect()
for _, row in df_views.iterrows():
    db_ex.execute(text(f'CREATE VIEW {row["table_schema"]+"."+row["table_name"]} AS {row['view_definition']}'))
db_ex.commit()